**Apache Spark - Introducción y Configuración**

**Objetivo**: Configurar Spark e iniciar el procesamiento distribuido de datos
masivos.

Configuracion de Spark session

In [77]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, sum as spark_sum

Configuramos un entorno local de Apache Spark utilizando SparkSession como punto de entrada principal.

In [78]:


# Crear una instancia de SparkSession
spark = SparkSession.builder \
    .appName("Retail Analytics Pipeline") \
    .master("local[*]") \
    .getOrCreate()


print("Spark session creada exitosamente: ", spark)



Spark session creada exitosamente:  <pyspark.sql.session.SparkSession object at 0x106f13d40>


In [79]:
sc = spark.sparkContext

print("Spark context creado exitosamente: ", sc)

Spark context creado exitosamente:  <SparkContext master=local[*] appName=Retail Analytics Pipeline>


Carga de los datasets iniciales correspondientes a órdenes, items y reseñas en formato RDD, validando su correcta lectura mediante acciones básicas como count() y take()

In [80]:
rdd_orders = sc.textFile("data/olist_orders_dataset.csv")

rdd_orders.take(5)

['"order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"',
 'e481f51cbdc54678b7cc49136f2d6af7,"9ef432eb6251297304e76186b10a928d",delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00',
 '"53cdb2fc8bc7dce0b6741e2150273451",b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00',
 '"47770eb9100c2d0c44946d9cf07ec65d","41ce2a54c0b03bf3443c3d931a367089",delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00',
 '"949d5b44dbf5de918fe9c16f97b45f8a",f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00']

In [81]:
rdd_reviews = sc.textFile("data/olist_order_reviews_dataset.csv")

rdd_reviews.take(5)


['"review_id","order_id","review_score","review_comment_title","review_comment_message","review_creation_date","review_answer_timestamp"',
 '"7bc2406110b926393aa56f80a40eba40","73fc7af87114b39712e6da79b0a377eb",4,,,2018-01-18 00:00:00,2018-01-18 21:46:59',
 '"80e641a11e56f04c1ad469d5645fdfde","a548910a1c6147796b98fdf73dbeba33",5,,,2018-03-10 00:00:00,2018-03-11 03:05:13',
 '"228ce5500dc1d8e020d8d1322874b6f0","f9e4b658b201a9f2ecdecbb34bed034b",5,,,2018-02-17 00:00:00,2018-02-18 14:36:24',
 '"e64fb393e7b32834bb789ff8bb30750e","658677c97b385a9be170737859d3511b",5,,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06']

In [82]:
rdd_items = sc.textFile("data/olist_order_items_dataset.csv")

rdd_items.take(5)

['"order_id","order_item_id","product_id","seller_id","shipping_limit_date","price","freight_value"',
 '"00010242fe8c5a6d1ba2dd792cb16214",1,"4244733e06e7ecb4970a6e2683c13e61","48436dade18ac8b2bce089ec2a041202",2017-09-19 09:45:35,58.90,13.29',
 '"00018f77f2f0320c557190d7a144bdd3",1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93',
 '"000229ec398224ef6ca0657da4fc703e",1,c777355d18b72b67abbeef9df44fd0fd,"5b51032eddd242adc84c38acab88f23d",2018-01-18 14:48:30,199.00,17.87',
 '"00024acbcdf0a6daa1e931b038114c75",1,"7634da152a4610f1595efa32f14722fc","9d7a1d34a5052409006425275ba1c2b4",2018-08-15 10:10:18,12.99,12.79']

In [83]:
print("cantidad de RDD de pedidos: ", rdd_orders.count())
print("cantidad de RDD de reviews: ", rdd_reviews.count())
print("cantidad de RDD de items: ", rdd_items.count())



cantidad de RDD de pedidos:  99442
cantidad de RDD de reviews:  104720
cantidad de RDD de items:  112651


**Elementos básicos de Spark (RDD, Transformaciones y Acciones)**

**Objetivo**: Manipular grandes volúmenes de datos mediante RDDs,
aplicando transformaciones y acciones.


Para realizar las operaciones y estructurar los datos eliminamos la primera file del dataset correspondiente al header de ordenes, para evitar que sea leido como un registro mas y evitar inconsistencias en le procesamiento.

In [84]:
#eliminamos el header de ordenes

header = rdd_orders.first()
rdd_orders = rdd_orders.filter(lambda x: x != header)

rdd_orders.take(3)

['e481f51cbdc54678b7cc49136f2d6af7,"9ef432eb6251297304e76186b10a928d",delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00',
 '"53cdb2fc8bc7dce0b6741e2150273451",b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00',
 '"47770eb9100c2d0c44946d9cf07ec65d","41ce2a54c0b03bf3443c3d931a367089",delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00']

A continuación, generamos estructuras más organizadas a partir de los datos, incluyendo la creación de Pair RDDs utilizando order_id como clave, facilitando operaciones posteriores.

In [85]:
pair_rdd = rdd_orders.map(lambda x: (
    x.split(",")[0].strip().strip('"'),
    x.split(",")[2].strip().strip('"')
))
pair_rdd.take(5)

[('e481f51cbdc54678b7cc49136f2d6af7', 'delivered'),
 ('53cdb2fc8bc7dce0b6741e2150273451', 'delivered'),
 ('47770eb9100c2d0c44946d9cf07ec65d', 'delivered'),
 ('949d5b44dbf5de918fe9c16f97b45f8a', 'delivered'),
 ('ad21c59c0840e6cb83a9ceb5573f8159', 'delivered')]

Ahora, aplicamos transformaciones como map, filter y distinct, lo que permite limpiar y estructurar los datos iniciales

Mediante la funcion map, extraemos la columna "order_status" desde cada fila del RDD.
Primero se separa cada registro por comas (split), luego se toma la posición [2], que corresponde al estado de la orden, y se eliminan espacios y comillas innecesarias.

El resultado es un nuevo RDD con solo los estados de las órdenes.
Al aplicar take(5), se muestran los primeros 5 valores, en este caso "delivered"

In [86]:
#map
order_status = rdd_orders.map(lambda x: x.split(",")[2].strip().strip('"'))
order_status.take(5)

['delivered', 'delivered', 'delivered', 'delivered', 'delivered']

Utilizamos filter para seleccionar únicamente las órdenes cuyo estado corresponde a delivered. Para ello, se extrae la columna order_status desde cada registro y se compara su valor, permitiendo trabajar solo con órdenes completadas

In [87]:
#filter
delivered_orders = rdd_orders.filter(
    lambda x: x.split(",")[2].strip().strip('"') == "delivered"
)
delivered_orders.take(5)

['e481f51cbdc54678b7cc49136f2d6af7,"9ef432eb6251297304e76186b10a928d",delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00',
 '"53cdb2fc8bc7dce0b6741e2150273451",b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00',
 '"47770eb9100c2d0c44946d9cf07ec65d","41ce2a54c0b03bf3443c3d931a367089",delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00',
 '"949d5b44dbf5de918fe9c16f97b45f8a",f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00',
 'ad21c59c0840e6cb83a9ceb5573f8159,"8ab97904e6daea8866dbdbc4fb7aad2c",delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00']

Ahora utilizamos flatMap para descomponer cada fila del dataset en sus elementos individuales, generando una lista plana con todos los valores de las columnas. Esto permite analizar los datos a nivel más granular

In [88]:
#flatmap
words = rdd_orders.flatMap(lambda x: x.split(","))
words.take(5)

['e481f51cbdc54678b7cc49136f2d6af7',
 '"9ef432eb6251297304e76186b10a928d"',
 'delivered',
 '2017-10-02 10:56:33',
 '2017-10-02 11:07:15']

usaremos distinct sobre el RDD de order_status para identificar los diferentes estados posibles de las órdenes, eliminando duplicados y obteniendo los valores únicos presentes en el dataset

In [89]:
#distinct
order_status.distinct().collect()

['canceled',
 'delivered',
 'invoiced',
 'shipped',
 'processing',
 'unavailable',
 'created',
 'approved']

Ahora ordenamos las órdenes en función de su identificador (order_id). Lo que nos permite observar los registros en un orden definido y validar la estructura de los datos

In [90]:
#sortBy
sorted_orders = rdd_orders.sortBy(lambda x: x.split(",")[0])
sorted_orders.take(5)

['"00010242fe8c5a6d1ba2dd792cb16214","3ce436f183e68e07877b285a838db11a",delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00',
 '"00018f77f2f0320c557190d7a144bdd3",f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00',
 '"000229ec398224ef6ca0657da4fc703e","6489ae5e4333f3693df5ad4372dab6d3",delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00',
 '"00024acbcdf0a6daa1e931b038114c75",d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00',
 '"00042b26cf59d7ce69dfabb4e55b4fd9","58dbd0b2d70206bf40e62cd34e84d795",delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00']

Creamos un Pair RDD utilizando order_status como clave y un valor constante de 1, con el objetivo de contar la cantidad de órdenes por estado. Posteriormente, se aplicamos reduceByKey para sumar los valores asociados a cada clave, obteniendo así el total de órdenes por cada tipo de estado presente en el dataset

In [91]:
# agregación: conteo de órdenes por estado usando Pair RDD
pair_rdd = rdd_orders.map(lambda x: (
    x.split(",")[2].strip().strip('"'),  # clave: order_status
    1                                    # valor: contador
))

# reducción por clave para obtener el total por estado
status_count = pair_rdd.reduceByKey(lambda a, b: a + b)

#resultado
status_count.collect()

[('canceled', 625),
 ('delivered', 96478),
 ('invoiced', 314),
 ('shipped', 1107),
 ('processing', 301),
 ('unavailable', 609),
 ('created', 5),
 ('approved', 2)]

**Lección 4**: Procesamiento de datos estructurados (Spark SQL y DataFrames)

**Objetivo**: Procesar y analizar datos estructurados de manera optimizada
usando DataFrames y Spark SQL.


En esta etapa se optó por convertir solo el dataset de órdenes desde RDD a DataFrame, con el objetivo de evitar problemas de consistencia en las otras tablas, especialmente en el caso de reseñas, donde ya se habían observado dificultades asociadas al manejo de texto libre y separaciones manuales por comas. Por esta razón, los datasets de items y reviews fueron cargados directamente como DataFrames mediante la lectura nativa de Spark.

A continuación, se trabajamos sobre el RDD de órdenes luego de haber eliminado el header previamente. Estructuramos el RDD separando cada fila en columnas mediante split(","). Luego, se seleccionamos tres columnas principales de interés: order_id, customer_id y order_status para trabajar con una estructura más simple y controlada para la conversión inicial

In [92]:
# Para convertir los RDDs a DataFrames utilizaremos el rdd cuyo header ya habia sido eliminado

# estructuramos el rdd
rdd_split = rdd_orders.map(lambda x: x.split(","))

# seleccionamos columnas necesarias, utilizaremos 3 columnas principales de nuestro interes
rdd_selected = rdd_split.map(lambda x: (
    x[0].strip('"'),
    x[1].strip('"'),
    x[2].strip('"')
))


definimos un esquema explícito, asignando nombre y tipo de dato a cada una de las columnas. Como resultado, Spark genera un DataFrame con tres campos de tipo string

In [93]:
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
])

In [94]:
# Creamos el dataframe

df_orders = spark.createDataFrame(rdd_selected, schema)

df_orders.show(5)

+--------------------+--------------------+------------+
|            order_id|         customer_id|order_status|
+--------------------+--------------------+------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered|
|949d5b44dbf5de918...|f88197465ea7920ad...|   delivered|
|ad21c59c0840e6cb8...|8ab97904e6daea886...|   delivered|
+--------------------+--------------------+------------+
only showing top 5 rows


In [95]:
#inspeccion de orders
df_orders.printSchema()

print("cantidad de pedidos: ", df_orders.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)



Traceback (most recent call last):
  File "<project>/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<project>/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


cantidad de pedidos:  99441


Realizamos una validación de los nulos

In [96]:
df_orders.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_orders.columns
]).show()

+--------+-----------+------------+
|order_id|customer_id|order_status|
+--------+-----------+------------+
|       0|          0|           0|
+--------+-----------+------------+



El análisis de valores nulos mostró que las columnas principales no presentan datos faltantes

In [97]:
total = df_orders.count()

df_orders.select([
    (spark_sum(col(c).isNull().cast("int")) / total).alias(c)
    for c in df_orders.columns
]).show()

+--------+-----------+------------+
|order_id|customer_id|order_status|
+--------+-----------+------------+
|     0.0|        0.0|         0.0|
+--------+-----------+------------+



Agrupamos por order status para verificar la consistencia de los datos

In [98]:
df_orders.groupBy("order_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



Cargamos el dataframe de reviews

In [99]:
df_reviews = spark.read.csv(
    "data/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

df_reviews.show(5)
print("cantidad de reviews: ", df_reviews.count())

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                NULL|                  NULL| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                NULL|                  NULL| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                NULL|  Recebi bem antes ...| 2017-04-21 00:00:00|   

Seleccionamos solo columnas relevantes

In [100]:
df_reviews = df_reviews.select(
    "order_id",
    "review_score"
)

Revsamos si hay nulos

In [101]:
total_reviews = df_reviews.count()

df_reviews.select([
    (spark_sum(col(c).isNull().cast("int")) / total_reviews).alias(c)
    for c in df_reviews.columns
]).show()

df_reviews = df_reviews.dropna(subset=["review_score"])

+--------+------------+
|order_id|review_score|
+--------+------------+
|     0.0|         0.0|
+--------+------------+



A continuación observamos que el review score se encuentra en rangos normales entre 1 y 5, order_id al ser alfanumerico no se considera.

In [102]:
df_reviews.describe().show()

+-------+--------------------+------------------+
|summary|            order_id|      review_score|
+-------+--------------------+------------------+
|  count|               99224|             99224|
|   mean|                NULL|  4.08642062404257|
| stddev|                NULL|1.3475791311150984|
|    min|00010242fe8c5a6d1...|                 1|
|    max|fffe41c64501cc87c...|                 5|
+-------+--------------------+------------------+



Luego cagramos el dataframe de items 

In [103]:
df_items = spark.read.csv("data/olist_order_items_dataset.csv", header=True, inferSchema=True)

df_items.show(5)

print("cantidad de items total: ", df_items.count())

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [104]:
print("cantidad de items con order_id distinto: ", df_items.select("order_id").distinct().count())

cantidad de items con order_id distinto:  98666


Filtramos solo columnas de interes

In [105]:
df_items = df_items.select(
    "order_id",
    "order_item_id",
    "product_id",
    "price",
    "freight_value"
)


A continuacion revisamos hacemos una inspeccion de nulos

In [106]:
total_items = df_items.count()

df_items.select([
    (spark_sum(col(c).isNull().cast("int")) / total_items).alias(c)
    for c in df_items.columns
]).show()

+--------+-------------+----------+-----+-------------+
|order_id|order_item_id|product_id|price|freight_value|
+--------+-------------+----------+-----+-------------+
|     0.0|          0.0|       0.0|  0.0|          0.0|
+--------+-------------+----------+-----+-------------+



In [107]:
df_items.describe().show()

+-------+--------------------+------------------+--------------------+------------------+------------------+
|summary|            order_id|     order_item_id|          product_id|             price|     freight_value|
+-------+--------------------+------------------+--------------------+------------------+------------------+
|  count|              112650|            112650|              112650|            112650|            112650|
|   mean|                NULL|1.1978339991122948|                NULL|120.65373901464174|19.990319928982977|
| stddev|                NULL|0.7051240313951721|                NULL| 183.6339280502595|15.806405412297098|
|    min|00010242fe8c5a6d1...|                 1|00066f42aeeb9f300...|              0.85|               0.0|
|    max|fffe41c64501cc87c...|                21|fffe9eeff12fcbd74...|            6735.0|            409.68|
+-------+--------------------+------------------+--------------------+------------------+------------------+



A continuación, se prueba una integración inicial mediante left join entre df_orders, df_items y df_reviews, utilizando order_id como clave común. Sin embargo, al comparar la cantidad de filas, se observó que el número de registros aumentaba desde 99.441 órdenes hasta 114.092 filas en el dataset unido. Además, se verificó que la tabla de items contenía 98.666 órdenes distintas. Este resultado mostró que el join directo generaba una expansión de filas, principalmente debido a la relación uno a muchos entre órdenes e items, lo que podía introducir inconsistencias si se quería mantener una fila por orden

In [108]:
df_joined = df_orders \
    .join(df_items, "order_id", "left") \
    .join(df_reviews, "order_id", "left")

df_joined.show(5)

Traceback (most recent call last):
  File "<project>/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<project>/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe
Traceback (most recent call last):
  File "<project>/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<project>/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


+--------------------+--------------------+------------+-------------+--------------------+-----+-------------+------------+
|            order_id|         customer_id|order_status|order_item_id|          product_id|price|freight_value|review_score|
+--------------------+--------------------+------------+-------------+--------------------+-----+-------------+------------+
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|            1|595fac2a385ac33a8...|118.7|        22.76|           4|
|949d5b44dbf5de918...|f88197465ea7920ad...|   delivered|            1|d0b61bfb1de832b15...| 45.0|         27.2|           5|
|ad21c59c0840e6cb8...|8ab97904e6daea886...|   delivered|            1|65266b2da20d04dbe...| 19.9|         8.72|           5|
|a233eaf2bd44e0532...|faa16af8d17f256f8...|   delivered|            1|457ca2b0e3bf044a6...| 90.0|        19.82|           5|
|981169eaa134f634b...|6d0ea4c9f46548807...|   delivered|            1|7d9b103535d14aebd...|  9.0|         7.78|           5|


In [109]:
print("cantidad de pedidos: ", df_orders.count())
print("cantidad de pedidos con reviews y items: ", df_joined.count())

cantidad de pedidos:  99441
cantidad de pedidos con reviews y items:  114092


In [110]:
print("cantidad de items diferetes", df_items.select("order_id").distinct().count())


cantidad de items diferetes 98666


Se optó por realizar una agregación previa de las tablas de items y reseñas antes de integrarlas. En el caso de items, se agruparon los registros por order_id para calcular el número total de productos (total_items), el precio total (total_price) y el costo total de envío (total_freight). En el caso de reviews, se agrupó por order_id para calcular el puntaje promedio de satisfacción (avg_review_score)

In [111]:
from pyspark.sql.functions import sum, avg, count

# Agregar items por orden
df_items_agg = df_items.groupBy("order_id").agg(
    count("order_item_id").alias("total_items"),
    sum("price").alias("total_price"),
    sum("freight_value").alias("total_freight")
)

# Agregar reviews por orden
df_reviews_agg = df_reviews.groupBy("order_id").agg(
    avg("review_score").alias("avg_review_score")
)

# Join final a nivel orden
df_final = df_orders \
    .join(df_items_agg, "order_id", "left") \
    .join(df_reviews_agg, "order_id", "left")

df_final.show(5)
print("cantidad de pedidos final: ", df_final.count())

+--------------------+--------------------+------------+-----------+-----------+-------------+----------------+
|            order_id|         customer_id|order_status|total_items|total_price|total_freight|avg_review_score|
+--------------------+--------------------+------------+-----------+-----------+-------------+----------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|          1|      29.99|         8.72|             4.0|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|          1|      118.7|        22.76|             4.0|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered|          1|      159.9|        19.22|             5.0|
|949d5b44dbf5de918...|f88197465ea7920ad...|   delivered|          1|       45.0|         27.2|             5.0|
|ad21c59c0840e6cb8...|8ab97904e6daea886...|   delivered|          1|       19.9|         8.72|             5.0|
+--------------------+--------------------+------------+-----------+-----------+-------------+----------

Traceback (most recent call last):
  File "<project>/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<project>/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


cantidad de pedidos final:  99441


In [112]:
df_final = df_final.filter(col("avg_review_score").isNotNull())

Creamos vista SQL

In [113]:
df_final.createOrReplaceTempView("orders_final")

calculamos el total de ventas sumando la columna total_price. Como resultado se obtuvo un valor cercano a 13.4 millones, lo que permite dimensionar el volumen total del negocio en el periodo analizado

In [114]:
spark.sql("""
SELECT SUM(total_price) AS total_sales
FROM orders_final
""").show()

+--------------------+
|         total_sales|
+--------------------+
|1.3466945000005752E7|
+--------------------+



Calculamos el promedio del puntaje de reseñas, obteniendo un valor aproximado de 4.08. Esto indica un nivel general de satisfacción alto por parte de los clientes

In [115]:
spark.sql("""
SELECT AVG(avg_review_score) AS avg_satisfaction
FROM orders_final
""").show()

+------------------+
|  avg_satisfaction|
+------------------+
|4.0867934152875325|
+------------------+



Se identificaron las órdenes con mayor valor económico, observándose pedidos que superan ampliamente el promedio, alcanzando montos superiores a 13.000. Esto evidencia la existencia de compras de alto valor dentro del dataset

In [116]:
top_orders = spark.sql("""
SELECT order_id, total_price
FROM orders_final
ORDER BY total_price DESC
LIMIT 10
""")

top_orders.show()

+--------------------+-----------+
|            order_id|total_price|
+--------------------+-----------+
|03caa2c082116e1d3...|    13440.0|
|736e1922ae60d0d6a...|     7160.0|
|0812eb902a67711a1...|     6735.0|
|f5136e38d1a14a4db...|     6499.0|
|2cc9089445046817a...|     5934.6|
|a96610ab360d42a2e...|     4799.0|
|199af31afc78c699f...|     4690.0|
|b4c4b76c642808cbe...|     4599.9|
|8dbc85d1447242f3b...|     4590.0|
|d2f270487125ddc41...|     4400.0|
+--------------------+-----------+



Ahora agrupamos las órdenes por estado, observándose que la gran mayoría corresponde a órdenes entregadas (más de 95.000). Esto indica un buen desempeño operativo, con pocos casos de cancelación o fallas

In [117]:
spark.sql("""
SELECT order_status, COUNT(*) as total_orders
FROM orders_final
GROUP BY order_status
ORDER BY total_orders DESC
""").show()

+------------+------------+
|order_status|total_orders|
+------------+------------+
|   delivered|       95832|
|     shipped|        1032|
|    canceled|         605|
| unavailable|         595|
|    invoiced|         309|
|  processing|         295|
|     created|           3|
|    approved|           2|
+------------+------------+



Se ordenaron las órdenes según su precio total para identificar las compras más altas del dataset. Como resultado, se observaron pedidos con valores considerablemente superiores al resto, destacando una orden de 13.440 y varias otras por sobre los 4.000. Esto confirma la existencia de un segmento de compras de alto valor dentro del negocio

In [118]:
spark.sql("""
SELECT order_id, total_price
FROM orders_final
ORDER BY total_price DESC
LIMIT 10
""").show()

+--------------------+-----------+
|            order_id|total_price|
+--------------------+-----------+
|03caa2c082116e1d3...|    13440.0|
|736e1922ae60d0d6a...|     7160.0|
|0812eb902a67711a1...|     6735.0|
|f5136e38d1a14a4db...|     6499.0|
|2cc9089445046817a...|     5934.6|
|a96610ab360d42a2e...|     4799.0|
|199af31afc78c699f...|     4690.0|
|b4c4b76c642808cbe...|     4599.9|
|8dbc85d1447242f3b...|     4590.0|
|d2f270487125ddc41...|     4400.0|
+--------------------+-----------+



Ahora analizamos el gasto promedio por nivel de satisfacción. Se observó que no existe una relación directa entre mayor gasto y mayor satisfacción, ya que algunos niveles bajos de reseña presentan incluso mayores valores de compra

In [119]:
spark.sql("""
SELECT avg_review_score, AVG(total_price) as avg_spent
FROM orders_final
GROUP BY avg_review_score
ORDER BY avg_review_score DESC
""").show()

+------------------+------------------+
|  avg_review_score|         avg_spent|
+------------------+------------------+
|               5.0|134.67466151005956|
|               4.5| 98.57870370370372|
| 4.333333333333333|             34.99|
|               4.0|132.48208100705588|
|               3.5|122.31040000000002|
|3.3333333333333335|              26.0|
|               3.0| 127.6379401212436|
|               2.5|107.83030303030304|
|               2.0|145.61364675154928|
|               1.5|           166.135|
|               1.0|166.97258675079178|
+------------------+------------------+



Calculamos el porcentaje de órdenes por nivel de reseña, observándose que aproximadamente un 57% de las órdenes tienen puntaje máximo (5), mientras que cerca de un 11% presentan la puntuación mínima (1). Esto muestra una distribución polarizada en la experiencia del cliente. El porcentaje se obtiene dividiendo el conteo de cada grupo entre el total de filas en `orders_final` mediante una subconsulta, sin usar funciones de ventana sobre todo el dataset.

In [120]:
spark.sql("""
SELECT 
    ROUND(avg_review_score, 2) as avg_review_score, 
    ROUND(AVG(total_price), 2) as avg_spent,
    COUNT(*) as total_orders,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders_final), 2) as percentage_orders
FROM orders_final
GROUP BY avg_review_score
ORDER BY avg_review_score DESC
""").show()

+----------------+---------+------------+-----------------+
|avg_review_score|avg_spent|total_orders|percentage_orders|
+----------------+---------+------------+-----------------+
|             5.0|   134.67|       56955|            57.72|
|             4.5|    98.58|          54|             0.05|
|            4.33|    34.99|           1|             0.00|
|             4.0|   132.48|       19018|            19.27|
|             3.5|   122.31|          25|             0.03|
|            3.33|     26.0|           1|             0.00|
|             3.0|   127.64|        8136|             8.25|
|             2.5|   107.83|          34|             0.03|
|             2.0|   145.61|        3125|             3.17|
|             1.5|   166.14|           8|             0.01|
|             1.0|   166.97|       11316|            11.47|
+----------------+---------+------------+-----------------+



Los datos procesados seran almacenados en formato Parquet, debido a que este formato columnar permite una lectura más eficiente y optimizada para análisis posteriores y modelos de Machine Learning. De esta manera, se deja preparado un dataset limpio y estructurado para la siguiente etapa del proyecto.

In [121]:
df_final.write.mode("overwrite").parquet("data/orders_final.parquet")

**Lección 5: Introducción a Machine Learning Escalable (Spark MLlib)**
**Objetivo**: Construir un pipeline de MLlib para clasificación y segmentación
de usuarios.


Para esta etapa trabajaremos sobre el dataset procesado y almacenado previamente en formato Parquet, asegurando continuidad entre el procesamiento estructurado y el pipeline de Machine Learning. A partir de esta base, se realizó una limpieza adicional enfocada en las variables que serían utilizadas para el modelamiento, filtrando registros con valores nulos en columnas relevantes

In [122]:
df_ml = spark.read.parquet("data/orders_final.parquet")
df_ml.show(5)
df_ml.printSchema()

+--------------------+--------------------+------------+-----------+-----------+-------------+----------------+
|            order_id|         customer_id|order_status|total_items|total_price|total_freight|avg_review_score|
+--------------------+--------------------+------------+-----------+-----------+-------------+----------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|          1|      29.99|         8.72|             4.0|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|          1|      118.7|        22.76|             4.0|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered|          1|      159.9|        19.22|             5.0|
|949d5b44dbf5de918...|f88197465ea7920ad...|   delivered|          1|       45.0|         27.2|             5.0|
|ad21c59c0840e6cb8...|8ab97904e6daea886...|   delivered|          1|       19.9|         8.72|             5.0|
+--------------------+--------------------+------------+-----------+-----------+-------------+----------

In [123]:
from pyspark.sql.functions import col

df_ml = df_ml.filter(
    col("total_items").isNotNull() &
    col("total_price").isNotNull() &
    col("total_freight").isNotNull() &
    col("order_status").isNotNull() &
    col("avg_review_score").isNotNull()
)

definimos una variable objetivo binaria asociada al nivel de satisfacción del cliente, construida a partir del puntaje promedio de reseñas para la regresión logistica

In [124]:
from pyspark.sql.functions import when

df_ml = df_ml.withColumn(
    "label",
    when(col("avg_review_score") >= 4, 1).otherwise(0)
)

transformamos la variable categórica order_status mediante StringIndexer para poder ser utilizada por los algoritmos de Spark MLlib

In [125]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="order_status",
    outputCol="order_status_index",
    handleInvalid="keep"
)

Creamos features con VectorAssembler

In [126]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["total_items", "total_price", "total_freight", "order_status_index"],
    outputCol="features"
)

Modelo supervisado: Regresión Logística

In [127]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

lr = LogisticRegression(featuresCol="features", labelCol="label")

pipeline_lr = Pipeline(stages=[indexer, assembler, lr])

train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

model_lr = pipeline_lr.fit(train_data)
predictions_lr = model_lr.transform(test_data)

predictions_lr.select("label", "prediction", "probability").show(5, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |1.0       |[0.19870118932144035,0.8012988106785597]|
|1    |1.0       |[0.20295125897314956,0.7970487410268504]|
|1    |1.0       |[0.19180533616420486,0.8081946638357951]|
|1    |1.0       |[0.19780199150906033,0.8021980084909397]|
|1    |1.0       |[0.19621547700368752,0.8037845229963125]|
+-----+----------+----------------------------------------+
only showing top 5 rows


Evaluamos regresión logística

In [128]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator_lr = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator_lr.evaluate(predictions_lr)
print("AUC:", auc)

AUC: 0.598555908463753


El resultado demuestra que el modelo tiene una capacidad predictiva limitada. Esto se debe principalmente a la simplicidad de las variables disponibles, ya que el modelo intenta predecir satisfacción utilizando solo variables transaccionales básicas. Factores más determinantes, como tiempos reales de entrega, calidad del producto o experiencia general del cliente, no estaban representados en el conjunto de features utilizado. Por lo tanto, aunque el modelo supera levemente el rendimiento aleatorio, no alcanza un nivel alto de discriminación.

A continuación entrenamos un modelo no supervisado, K-Means,  con el objetivo de segmentar órdenes según comportamiento de compra y satisfacción.

In [129]:
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

pipeline_kmeans = Pipeline(stages=[indexer, assembler])

df_features = pipeline_kmeans.fit(df_ml).transform(df_ml)

kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=3, seed=42)
model_kmeans = kmeans.fit(df_features)
predictions_kmeans = model_kmeans.transform(df_features)

predictions_kmeans.select("order_id", "cluster", "total_price", "avg_review_score").show(10)

+--------------------+-------+-----------+----------------+
|            order_id|cluster|total_price|avg_review_score|
+--------------------+-------+-----------+----------------+
|e481f51cbdc54678b...|      0|      29.99|             4.0|
|53cdb2fc8bc7dce0b...|      0|      118.7|             4.0|
|47770eb9100c2d0c4...|      0|      159.9|             5.0|
|949d5b44dbf5de918...|      0|       45.0|             5.0|
|ad21c59c0840e6cb8...|      0|       19.9|             5.0|
|a4591c265e18cb1dc...|      0|      147.9|             4.0|
|136cce7faa42fdb2c...|      0|       49.9|             2.0|
|6514b8ad8028c9f2c...|      0|      59.99|             5.0|
|76c6e866289321a7c...|      0|       19.9|             1.0|
|e69bfb5eb88e0ed6a...|      0|     149.99|             5.0|
+--------------------+-------+-----------+----------------+
only showing top 10 rows


Evaluamos K-Means

In [130]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator_kmeans = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette"
)

silhouette = evaluator_kmeans.evaluate(predictions_kmeans)
print("Silhouette Score:", silhouette)

Silhouette Score: 0.9034190762875656


El modelo de K-Means obtuvo un Silhouette Score de aproximadamente 0.90, lo que indica una separación muy clara entre los clusters generados. Este resultado sugiere que los grupos identificados presentan alta coherencia interna y una buena diferenciación entre sí.

Sin embargo, este alto valor también se explica por la naturaleza de las variables utilizadas, especialmente el precio total de las órdenes, que presenta una alta dispersión y permite una segmentación clara de los datos.

A continuacion separamos los ccluster mediante SQL para generar insights

In [131]:
predictions_kmeans.groupBy("cluster").avg(
    "total_price", "total_items", "total_freight", "avg_review_score"
).show()

+-------+------------------+------------------+------------------+---------------------+
|cluster|  avg(total_price)|  avg(total_items)|avg(total_freight)|avg(avg_review_score)|
+-------+------------------+------------------+------------------+---------------------+
|      1| 512.2762380818775|1.3900729108244532| 44.95339876612448|   3.9677509814918674|
|      2|1740.5121625163836|1.4141546526867628|  75.0172346002621|    3.891218872870249|
|      0|   94.258959920907|1.1187820754926574| 20.60668303303533|   4.1177452918916115|
+-------+------------------+------------------+------------------+---------------------+



- Como resultado, se identificaron tres clusters con perfiles diferenciados. El primer grupo presentó un gasto promedio muy alto y costos de envío elevados, pero con un nivel de satisfacción promedio inferior al de los otros grupos. El segundo cluster mostró un comportamiento intermedio, tanto en gasto como en satisfacción. Finalmente, el tercer grupo correspondió a órdenes de bajo valor económico, pero con el puntaje promedio de satisfacción más alto.

- Los clientes con mayor gasto tienden a presentar una satisfacción algo menor que aquellos con compras más pequeñas. A partir de ello, se interpretó que los segmentos de alto valor pueden tener expectativas más altas respecto a la experiencia de compra, lo que los convierte en un grupo especialmente sensible para estrategias de 

- En términos de marketing, los resultados del clustering sugieren que el segmento de alto gasto debería ser priorizado en acciones orientadas a mejorar experiencia, logística y postventa, mientras que los segmentos de menor valor pero alta satisfacción podrían representar oportunidades de crecimiento mediante estrategias de upselling o fidelización.
